# Install dependencies
%pip install langchain langchain-core langchain-text-splitters langchain-community langchain-ollama langchain-experimental neo4j python-dotenv

In [ ]:
import json
import os
from pathlib import Path
from dotenv import load_dotenv

# LangChain imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.graphs import Neo4jGraph
from langchain_community.vectorstores import Neo4jVector
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from pydantic import BaseModel, Field

load_dotenv()

print("✓ Dependencies loaded")

## 1. Cấu hình và Kết nối

In [ ]:
# Configuration
NEO4J_URL = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "changeme"
OLLAMA_BASE = "http://localhost:11434"
LLM_MODEL = "llama3.1:8b"
EMBED_MODEL = "nomic-embed-text"

# Paths
REPO_ROOT = Path().resolve().parent
DATA_PATH = REPO_ROOT / "graphrag" / "input" / "medical_reference_vi_qa.json"

print(f"Data path: {DATA_PATH}")
print(f"LLM Model: {LLM_MODEL}")

In [ ]:
# Test connections
try:
    graph = Neo4jGraph(
        url=NEO4J_URL,
        username=NEO4J_USER,
        password=NEO4J_PASSWORD,
    )
    print("✓ Neo4j connected")
except Exception as e:
    print(f"✗ Neo4j error: {e}")

# Test Ollama
import requests
try:
    r = requests.get(f"{OLLAMA_BASE}/api/tags")
    if r.status_code == 200:
        models = [m["name"] for m in r.json().get("models", [])]
        print(f"✓ Ollama connected. Models: {models}")
    else:
        print(f"✗ Ollama status: {r.status_code}")
except Exception as e:
    print(f"✗ Ollama error: {e}")

## 2. Load và Chuẩn bị Data

In [ ]:
MAX_RECORDS = 3000

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)[:MAX_RECORDS]

print(f"Loaded {len(data)} records")

# Convert to LangChain Documents
# NOTE: build graph from answer-focused medical facts instead of Q&A text to reduce pseudo edges.
documents = []
for rec in data:
    content = (rec.get("content", "") or "").strip()
    title = (rec.get("title", "") or "").strip()

    if not content:
        continue

    doc = Document(
        page_content=content,
        metadata={
            "title": title,
            "question": title,
            "source_url": rec.get("source_url", ""),
            "source_org": rec.get("source_org", ""),
            "topic_id": rec.get("topic_id", ""),
        }
    )
    documents.append(doc)

print(f"Created {len(documents)} documents")
print(f"\nExample:")
print(documents[0].page_content[:300])

In [ ]:
# Chunk documents
# Larger chunks keep medical relations in one span, reducing fragmented pseudo links.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=120,
    separators=["\n\n", "\n", ". ", "; ", ", ", " ", ""],
)

chunks = text_splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks")

## 3. Extract Graph (Bước chậm nhất)

In [ ]:
# Setup LLM
llm = ChatOllama(
    model=LLM_MODEL,
    base_url=OLLAMA_BASE,
    temperature=0,
    format="json",
)

In [ ]:
# Graph transformer với strict mode để hạn chế quan hệ/nút ngoài schema.
llm_transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=[
        "Disease",
        "Symptom",
        "Drug",
        "Treatment",
        "BodyPart",
        "Test",
        "RiskFactor",
        "Cause"],
    allowed_relationships=[
        "HAS_SYMPTOM",        # Disease → Symptom
        "TREATED_BY",         # Disease → Drug/Treatment
        "AFFECTS",            # Disease → BodyPart
        "DIAGNOSED_BY",       # Disease → Test
        "HAS_CAUSE",          # Disease → Cause
        "HAS_RISK_FACTOR",    # Disease → RiskFactor
    ],
    strict_mode=True,
)

print(f"✓ LLM ready: {LLM_MODEL}")

In [ ]:
import time
import pickle
from pathlib import Path

# ============================================
# ⚙️ CẤU HÌNH RESUME VÀ BATCH SIZE
# ============================================

BATCH_SIZE = 2  # Giảm xuống 2 để tránh quá tải
START_BATCH = 1380  # Đổi thành batch muốn resume (ví dụ: 300)

# File để lưu progress (tùy chọn)
PROGRESS_FILE = Path("graph_progress.pkl")

# ============================================
# 🚀 HÀM HỖ TRỢ
# ============================================

def save_progress(batch_num, processed_count):
    """Lưu progress để resume sau"""
    progress = {
        "last_batch": batch_num,
        "processed": processed_count,
        "timestamp": time.time()
    }
    with open(PROGRESS_FILE, "wb") as f:
        pickle.dump(progress, f)
    print(f"  💾 Saved progress: batch {batch_num}")

def load_progress():
    """Đọc progress đã lưu"""
    if PROGRESS_FILE.exists():
        with open(PROGRESS_FILE, "rb") as f:
            return pickle.load(f)
    return {"last_batch": 0, "processed": 0}

def clear_progress():
    """Xóa progress file"""
    if PROGRESS_FILE.exists():
        PROGRESS_FILE.unlink()

# ============================================
# 🧪 TEST NHỎ TRƯỚC (KHUYẾN NGHỊ)
# ============================================

# Bỏ comment dòng này để test với 50 chunks đầu tiên
# chunks = chunks[:50]
# print(f"⚠️ TEST MODE: Chỉ xử lý {len(chunks)} chunks")

# ============================================
# 🔄 MAIN PROCESSING VỚI RESUME
# ============================================

# Kiểm tra resume từ file (hoặc dùng START_BATCH cứng)
saved = load_progress()
if saved["last_batch"] > 0 and START_BATCH == 0:
    resume_from = saved["last_batch"]
    print(f"🔄 Resuming from saved batch {resume_from}")
else:
    resume_from = START_BATCH
    if resume_from > 0:
        print(f"🔄 Starting from batch {resume_from} (manual)")
    else:
        print("🔄 Starting from beginning")
        # Clear existing graph only nếu chạy từ đầu
        graph.query("MATCH (n) DETACH DELETE n")
        print("✓ Cleared existing graph")
        clear_progress()

# Tính start index
total_chunks = len(chunks)
start_index = resume_from * BATCH_SIZE

ALLOWED_RELATIONSHIPS = {
    "HAS_SYMPTOM",
    "TREATED_BY",
    "AFFECTS",
    "DIAGNOSED_BY",
    "HAS_CAUSE",
    "HAS_RISK_FACTOR",
}


def _clean_graph_docs(graph_docs):
    """Filter low-value graph artifacts before writing to Neo4j."""
    cleaned = []
    for gd in graph_docs:
        rels = []
        for rel in getattr(gd, "relationships", []) or []:
            rel_type = (getattr(rel, "type", "") or "").strip()
            src = (getattr(getattr(rel, "source", None), "id", "") or "").strip().lower()
            tgt = (getattr(getattr(rel, "target", None), "id", "") or "").strip().lower()
            if rel_type not in ALLOWED_RELATIONSHIPS:
                continue
            if not src or not tgt or src == tgt:
                continue
            rels.append(rel)
        gd.relationships = rels

        # Keep doc only when it has at least one relation and two nodes.
        nodes = list(getattr(gd, "nodes", []) or [])
        if len(nodes) >= 2 and len(rels) > 0:
            cleaned.append(gd)
    return cleaned


if start_index >= total_chunks:
    print(f"✓ All {total_chunks} chunks already processed!")
else:
    print(f"📊 Total: {total_chunks} chunks, Batch size: {BATCH_SIZE}")
    print(f"📊 Resuming from chunk {start_index} (batch {resume_from})")
    print(f"📊 Remaining: {total_chunks - start_index} chunks\n")

    processed_count = saved.get("processed", 0) if resume_from > 0 else 0
    total_batches = (total_chunks - 1) // BATCH_SIZE + 1

    # Main loop
    for i in range(start_index, total_chunks, BATCH_SIZE):
        batch_num = i // BATCH_SIZE
        batch = chunks[i:i + BATCH_SIZE]

        print(f"[{batch_num + 1}/{total_batches}] Processing {len(batch)} chunks... ", end="", flush=True)

        start_time = time.time()

        try:
            # Extract graph documents
            graph_docs = llm_transformer.convert_to_graph_documents(batch)
            graph_docs = _clean_graph_docs(graph_docs)

            if not graph_docs:
                print("⚠️ No high-quality entities/relations found")
                continue

            # Add to Neo4j
            graph.add_graph_documents(
                graph_docs,
                baseEntityLabel=True,
                include_source=True,
            )

            elapsed = time.time() - start_time
            processed_count += len(batch)

            # Save progress mỗi 5 batches
            if (batch_num + 1) % 5 == 0:
                save_progress(batch_num + 1, processed_count)

            print(f"✓ ({elapsed:.1f}s) - {len(graph_docs)} docs")

        except Exception as e:
            print(f"\n  ❌ Error at batch {batch_num}: {e}")
            save_progress(batch_num, processed_count)
            print(f"  💾 Progress saved. Fix error và chạy lại để resume.")
            raise  # Dừng lại để user xử lý lỗi

    # Xóa progress khi hoàn thành
    clear_progress()
    print(f"\n✅ Done! Processed {processed_count} chunks total")

In [ ]:
# Embeddings
embeddings = OllamaEmbeddings(
    model=EMBED_MODEL,
    base_url=OLLAMA_BASE,
)

# Create vector index from existing Document nodes
vector_index = Neo4jVector.from_existing_graph(
    embeddings,
    url=NEO4J_URL,
    username=NEO4J_USER,
    password=NEO4J_PASSWORD,
    search_type="hybrid",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding",
)

# Ensure fulltext indexes used by retrievers exist
graph.query("""
CREATE FULLTEXT INDEX documentFulltext IF NOT EXISTS
FOR (d:Document) ON EACH [d.text]
""")

graph.query("""
CREATE FULLTEXT INDEX graphEntityFulltext IF NOT EXISTS
FOR (n:__Entity__) ON EACH [n.id]
""")

vector_retriever = vector_index.as_retriever(search_kwargs={"k": 4})
print("✓ Vector index + fulltext indexes ready")

In [ ]:
class Entities(BaseModel):
    """Extract entities from question."""
    names: list[str] = Field(
        ...,
        description="Danh sách thực thể y khoa trọng tâm trong câu hỏi",
    )

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Bạn là chuyên gia trích xuất thực thể y khoa tiếng Việt. "
        "Chỉ lấy thực thể quan trọng để truy vấn knowledge graph. "
        "Không lấy từ chung chung như 'bệnh', 'thuốc', 'cao'.",
    ),
    (
        "human",
        "Câu hỏi: {question}\n"
        "Trả về JSON với key names (list[str]) gồm thực thể cụ thể như bệnh, thuốc, triệu chứng, xét nghiệm.",
    ),
])

entity_chain = prompt | llm.with_structured_output(Entities)


def normalize_entities(names: list[str]) -> list[str]:
    out = []
    seen = set()
    for n in names or []:
        x = " ".join((n or "").strip().lower().split())
        if not x:
            continue
        if x in {"bệnh", "thuốc", "triệu chứng", "cao", "thấp", "điều trị"}:
            continue
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out


# Test
test = entity_chain.invoke({"question": "Tôi bị sốt 38 độ, đau đầu và ho?"})
print(f"Entities: {normalize_entities(test.names)}")

## 6. Graph Retriever

In [ ]:
def _entity_terms(entity: str) -> list[str]:
    e = " ".join((entity or "").strip().lower().split())
    if not e:
        return []
    out = [e]
    for prefix in ["bệnh ", "triệu chứng ", "thuốc "]:
        if e.startswith(prefix):
            t = e[len(prefix):].strip()
            if t:
                out.append(t)
    if " " in e:
        out.append(e.split()[0])
    dedup = []
    seen = set()
    for t in out:
        if len(t) >= 2 and t not in seen:
            seen.add(t)
            dedup.append(t)
    return dedup


def graph_retriever(question: str) -> str:
    """Retrieve relevant graph context with fulltext first, then fallback matching."""
    entities = entity_chain.invoke({"question": question})
    names = normalize_entities(entities.names)

    if not names:
        return ""

    relation_types = [
        "HAS_SYMPTOM",
        "HAS_CAUSE",
        "HAS_RISK_FACTOR",
        "DIAGNOSED_BY",
        "TREATED_BY",
        "AFFECTS",
    ]

    rows = []
    seen_rows = set()

    for entity in names:
        terms = _entity_terms(entity)

        for term in terms:
            try:
                response = graph.query(
                    """
                    CALL db.index.fulltext.queryNodes('graphEntityFulltext', $term)
                    YIELD node, score
                    WITH node, score ORDER BY score DESC LIMIT 5

                    OPTIONAL MATCH (node)-[r]->(neighbor)
                    WHERE type(r) IN $relation_types

                    RETURN
                      coalesce(node.id, '') AS src,
                      type(r) AS rel,
                      coalesce(neighbor.id, '') AS dst,
                      score
                    LIMIT 20
                    """,
                    {"term": term, "relation_types": relation_types},
                )
            except Exception:
                response = []

            if not response:
                response = graph.query(
                    """
                    MATCH (n:__Entity__)
                    WHERE toLower(coalesce(n.id, '')) CONTAINS $term
                    WITH n LIMIT 5
                    OPTIONAL MATCH (n)-[r]->(neighbor)
                    WHERE type(r) IN $relation_types
                    RETURN
                      coalesce(n.id, '') AS src,
                      type(r) AS rel,
                      coalesce(neighbor.id, '') AS dst,
                      0.0 AS score
                    LIMIT 20
                    """,
                    {"term": term.lower(), "relation_types": relation_types},
                )

            for rec in response:
                src = (rec.get("src") or "").strip()
                rel = (rec.get("rel") or "").strip()
                dst = (rec.get("dst") or "").strip()
                if not src:
                    continue
                line = f"{src} - {rel} -> {dst}" if rel and dst else src
                if line not in seen_rows:
                    seen_rows.add(line)
                    rows.append(line)

            if len(rows) >= 40:
                break
        if len(rows) >= 40:
            break

    return "\n".join(rows[:40])


# Test
print("Testing graph retriever...")
test_result = graph_retriever("Sốt cao là triệu chứng của bệnh gì?")
print(f"Graph context:\n{test_result if test_result else '(No results)'}")

## 7. Full RAG Chain

In [ ]:
def full_retriever(question: str) -> str:
    graph_data = graph_retriever(question)

    # ưu tiên graph; vector chỉ để bù phần thiếu
    vector_docs = vector_retriever.invoke(question)[:4]
    vector_data = "\n\n".join(d.page_content for d in vector_docs if getattr(d, "page_content", ""))

    if not graph_data:
        graph_data = "(Không tìm thấy quan hệ trực tiếp trong graph cho câu hỏi này)"

    if not vector_data:
        vector_data = "(Không có ngữ cảnh vector phù hợp)"

    return f"""
NGUỒN CHÍNH (Knowledge Graph):
{graph_data}

NGUỒN PHỤ (Vector Search):
{vector_data}
""".strip()


# Prompt template

template = """
Bạn là trợ lý hỗ trợ hỏi đáp y khoa.

Quy tắc:
1. Ưu tiên dùng Knowledge Graph.
2. Chỉ dùng Vector Search để bổ sung nếu graph thiếu dữ liệu.
3. Nếu hai nguồn mâu thuẫn, ưu tiên Knowledge Graph.
4. Không suy diễn ngoài dữ liệu cung cấp.
5. Nếu hỏi từ triệu chứng -> bệnh, chỉ nêu khả năng có thể, không kết luận chắc chắn.
6. Nếu dữ liệu không đủ, nói rõ "Không có đủ thông tin trong dữ liệu đã truy xuất".

Ngữ cảnh:
{context}

Câu hỏi:
{question}

Trả lời ngắn gọn, chính xác bằng tiếng Việt.
"""

prompt = ChatPromptTemplate.from_template(template)

# Chain
chain = (
    {
        "context": full_retriever,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("✓ RAG chain ready")


## 8. Test Queries

In [ ]:
# Test 1
q1 = "Tôi sốt 38 độ, đau đầu và mệt mỏi - có thể là bệnh gì?"
print(f"Q: {q1}")
print(f"A: {chain.invoke(q1)}")
print("\n" + "="*50 + "\n")

# Test 2
q2 = "Thuốc paracetamol dùng để làm gì?"
print(f"Q: {q2}")
print(f"A: {chain.invoke(q2)}")